In [ ]:
import pandas as pd
import re

csv_path = "../../ALL 5 YRS PAPER.csv"
df = pd.read_csv(csv_path)
print(df.columns)

In [ ]:
ml_models = [
    "SVM", "Support Vector Machine", "Support Vector Machines",
    "Random Forest", "RandomForest", "Random Forests",
    "Decision Tree", "Decision Trees",
    "XGBoost", "Gradient Boosting", "GBM", "LightGBM", "CatBoost",
    "KNN", "K-Nearest Neighbor", "K Nearest Neighbors",
    "LASSO", "Elastic Net", "Ridge Regression", "Logistic Regression", "Linear Regression",
    "CNN", "Convolutional Neural Network", "Convolutional Networks",
    "DNN", "DNNs",
    "Autoencoder", "Auto-Encoder",
    "Neural Network", "Neural Networks",
    "Deep Learning", "Deep-Learning",
    "Ensemble Learning"
]


data = []
for index, row in df.iterrows():
    title = str(row.get("Title", ""))
    abstract = str(row.get("Abstract Note", ""))
    
    year = row.get("Publication Year", "")
    authors = row.get("Author", "")
    journal = row.get("Publication Title", "")
    
    data_source = ""
    matches = re.findall(r"plasma|urine|serum|tissue|cell line|cohort|hospital|dataset", abstract, re.I)
    if matches:
        data_source = ", ".join(sorted(set([x.lower() for x in matches])))
    
    cohort_size = ""
    match = re.search(r"(\d{1,5})\s*(patients|subjects|volunteers|cases|samples|participants)", abstract, re.I)
    if match:
        cohort_size = match.group(1)
    
    models_found = [m for m in ml_models if re.search(r"\b"+re.escape(m)+r"\b", abstract, re.I)]
    model_found = ", ".join(sorted(set(models_found)))
    
    data.append([year, title, authors, journal, data_source, cohort_size, model_found])

output_path = "D:/LS/20251020-LS-literature_summary.csv"
df_out = pd.DataFrame(data, columns=["Year","Title","Author","Journal","Data source","Cohort size","Machine Learning Model"])
df_out.to_csv(output_path, index=False)

print(f"saved at {output_path}")
print(f"Matched {df_out[df_out['Machine Learning Model']!=''].shape[0]} / {len(df)} documents "
      f"({100 * df_out[df_out['Machine Learning Model']!=''].shape[0]/len(df):.1f}% contain identifiable models)")


In [ ]:
# Step 1: year - count
import matplotlib.pyplot as plt

year_count = df['Year'].value_counts().sort_index()

plt.figure(figsize=(9,6))
plt.plot(year_count.index, year_count.values, marker='o', markersize=8, linewidth=2, color='#1f77b4')
plt.title("Number of Publications per Year", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Number of Publications", fontsize=14)
plt.xticks(year_count.index, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

for x, y in zip(year_count.index, year_count.values):
    plt.text(x, y+5, str(y), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Step 2: model - count
import re
df_models = df[['Machine Learning Model']].dropna()

df_models = df_models.assign(**{
    'Machine Learning Model': df_models['Machine Learning Model'].str.split(',')
}).explode('Machine Learning Model')

df_models['Machine Learning Model'] = (
    df_models['Machine Learning Model']
    .str.strip()
    .str.lower()
)

def normalize_model(name):
    if not isinstance(name, str):
        return name
    name = re.sub(r'\s+', ' ', name)  
    
    mapping = {
        'randomforest': 'random forest',
        'random forest': 'random forest',
        'svm': 'svm',
        'support vector machine': 'svm',
        'cnn': 'cnn',
        'convolutional neural network': 'cnn',
        'lasso': 'lasso',
        'logistic regression': 'logistic regression',
        'decision tree': 'decision tree',
        'xgboost': 'xgboost',
        'gradient boosting': 'gradient boosting',
        'autoencoder': 'autoencoder',
        'deep learning': 'deep learning',
        'neural network': 'neural network'
    }
    for key, val in mapping.items():
        if name == key:
            return val
    return name

df_models['Machine Learning Model'] = df_models['Machine Learning Model'].apply(normalize_model)

model_count = df_models['Machine Learning Model'].value_counts()

plt.figure(figsize=(10,6))
model_count.sort_values(ascending=True).plot(kind='barh', color='#4C72B0')
plt.title("Machine Learning Model Frequency", fontsize=16)
plt.xlabel("Count", fontsize=14)
plt.ylabel("Model", fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Top Model Co-occurrence Matrix
df_models = df_models.assign(**{
    'Machine Learning Model': df_models['Machine Learning Model'].str.split(',')
}).explode('Machine Learning Model')

df_models['Machine Learning Model'] = (
    df_models['Machine Learning Model']
    .str.strip()
    .str.lower()
)

model_count = df_models['Machine Learning Model'].value_counts()
top_models = model_count.head(10)
fig, axes = plt.subplots(2, 1, figsize=(10, 10), gridspec_kw={'height_ratios':[1,1.2]})


palette = sns.color_palette("YlGnBu", n_colors=len(top_models))
top_vals = top_models.sort_values(ascending=False)
sns.barplot(
    x=top_vals.values,
    y=top_vals.index,
    ax=axes[0],
    palette=palette[::-1]
)
axes[0].set_title("Machine Learning Model Frequency", fontsize=14, weight='bold')
axes[0].set_xlabel("Count", fontsize=12)
axes[0].set_ylabel("Model", fontsize=12)
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

for i, v in enumerate(top_vals.values):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=10)

sns.heatmap(
    matrix, annot=True, fmt="d",
    cmap="YlGnBu",
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Co-occurrence'},
    ax=axes[1]
)
axes[1].set_title("Top Model Co-occurrence Matrix", fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Year × Model 
df_long = df[['Year', 'Machine Learning Model']].dropna()

df_long = df_long.assign(**{
    'Machine Learning Model': df_long['Machine Learning Model'].str.split(',')
}).explode('Machine Learning Model')
df_long['Machine Learning Model'] = df_long['Machine Learning Model'].str.strip().str.lower()

mapping = {
    'svm': 'SVM',
    'support vector machine': 'SVM',
    'support vector machines': 'SVM',
    'random forest': 'Random Forest',
    'random forests': 'Random Forest',
    'decision tree': 'Decision Tree',
    'decision trees': 'Decision Tree',
    'xgboost': 'XGBoost',
    'gradient boosting': 'Gradient Boosting',
    'gbm': 'XGBoost',
    'lightgbm': 'XGBoost',
    'catboost': 'CatBoost',
    'knn': 'KNN',
    'k-nearest neighbor': 'KNN',
    'k nearest neighbors': 'KNN',
    'lasso': 'LASSO',
    'elastic net': 'Elastic Net',
    'ridge regression': 'Ridge Regression',
    'logistic regression': 'Logistic Regression',
    'linear regression': 'Linear Regression',
    'cnn': 'CNN',
    'convolutional neural network': 'CNN',
    'convolutional networks': 'CNN',
    'dnn': 'DNN',
    'dnns': 'DNN',
    'autoencoder': 'Autoencoder',
    'auto-encoder': 'Autoencoder',
    'neural network': 'Neural Network',
    'neural networks': 'Neural Network',
    'deep learning': 'Deep Learning',
    'deep-learning': 'Deep Learning',
    'ensemble learning': 'Ensemble Learning'
}

df_long['Model Standard'] = df_long['Machine Learning Model'].map(mapping)
df_long = df_long.dropna(subset=['Model Standard'])

tml_models = ['SVM', 'Random Forest', 'Decision Tree', 'XGBoost', 'Gradient Boosting', 
              'KNN', 'LASSO', 'Elastic Net', 'Ridge Regression', 'Logistic Regression', 'Linear Regression', 'Ensemble Learning']
dl_models = ['CNN', 'DNN', 'Autoencoder', 'Neural Network', 'Deep Learning', 'CatBoost']

df_long['Category'] = df_long['Model Standard'].apply(
    lambda x: 'TML' if x in tml_models else ('DL' if x in dl_models else 'Other')
)

pivot_table = pd.pivot_table(
    df_long,
    index='Year',
    columns='Model Standard',
    aggfunc='size',
    fill_value=0
).sort_index()

tml_colors = sns.color_palette("Blues", n_colors=len([m for m in pivot_table.columns if m in tml_models]))
dl_colors = sns.color_palette("YlOrBr", n_colors=len([m for m in pivot_table.columns if m in dl_models]))
colors = []
for col in pivot_table.columns:
    if col in tml_models:
        colors.append(tml_colors[tml_models.index(col)])
    elif col in dl_models:
        colors.append(dl_colors[dl_models.index(col)])
    else:
        colors.append('grey') 

pivot_table[tml_models + dl_models].plot(
    kind='bar',
    stacked=True,
    figsize=(14,6),
    color=colors
)

plt.title("Machine Learning Model Usage per Year", fontsize=16, weight='bold')
plt.xlabel("Year", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Model", bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: TML / DL
import pandas as pd
import matplotlib.pyplot as plt

tml_models = ['SVM', 'Random Forest', 'XGBoost', 'LASSO', 'Logistic Regression', 'Gradient Boosting', 'Decision Tree', 'RandomForest']
dl_models = ['CNN', 'Autoencoder', 'Neural Network', 'Deep Learning']

df_long['Model Type'] = df_long['Machine Learning Model'].apply(
    lambda x: 'DL' if x in dl_models else ('TML' if x in tml_models else 'Other')
)

type_pivot = df_long.pivot_table(
    index='Year',
    columns='Model Type',
    aggfunc='size',
    fill_value=0
).sort_index()


type_pivot[['TML','DL']].plot(kind='bar', stacked=True, figsize=(10,6), color=['lightcoral','skyblue'])
plt.title("Traditional ML vs Deep Learning Publications per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.legend(title="Model Type")
plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Proportion of TML vs DL Models (All Years)
df_models = df_models.assign(**{
    'Machine Learning Model': df_models['Machine Learning Model'].str.split(',')
}).explode('Machine Learning Model')
df_models['Machine Learning Model'] = df_models['Machine Learning Model'].str.strip().str.lower()

mapping = {
    'randomforest': 'random forest',
    'random forest': 'random forest',
    'svm': 'svm',
    'support vector machine': 'svm',
    'support vector machines': 'svm',
    'cnn': 'cnn',
    'convolutional neural network': 'cnn',
    'convolutional networks': 'cnn',
    'lasso': 'lasso',
    'logistic regression': 'logistic regression',
    'decision tree': 'decision tree',
    'decision trees': 'decision tree',
    'xgboost': 'xgboost',
    'gradient boosting': 'gradient boosting',
    'gbm': 'xgboost',
    'lightgbm': 'xgboost',
    'catboost': 'catboost',
    'autoencoder': 'autoencoder',
    'deep learning': 'deep learning',
    'deep-learning': 'deep learning',
    'dnn': 'deep learning',
    'dnns': 'deep learning',
    'neural network': 'deep learning',
    'neural networks': 'deep learning'
}

df_models['Model_Normalized'] = df_models['Machine Learning Model'].apply(lambda x: mapping.get(x, x))

tml_models = ['svm', 'random forest', 'decision tree', 'xgboost', 'catboost', 'lasso', 'logistic regression', 'autoencoder']
dl_models = ['cnn', 'deep learning']

df_models['Type'] = df_models['Model_Normalized'].apply(lambda x: 'TML' if x in tml_models else ('DL' if x in dl_models else 'Other'))

type_counts = df_models['Type'].value_counts()
type_counts = type_counts[['TML','DL']]

colors = ['#a7d3f1','#f1e1a7']  

plt.figure(figsize=(7,7))
plt.pie(
    type_counts,
    labels=type_counts.index,
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops={'width':0.5, 'edgecolor':'w'},
    autopct='%1.1f%%'
)
plt.title("Proportion of TML vs DL Models (All Years)", fontsize=14, weight='bold')
plt.tight_layout()
plt.show()
